In [ ]:
import re
import requests
from bs4 import BeautifulSoup
import json
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import List, Dict, Any

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://www.avito.ru/',
}

session = requests.Session()
session.cookies.set('_ga', 'dummy')
session.cookies.set('_ym_uid', 'dummy')

url = input('Ссылка на объявление: ').strip()

try:
    resp = session.get(url, headers=headers, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    # ID
    id_span = soup.find('span', {'data-marker': 'item-view/item-id'})
    if id_span:
        text = id_span.get_text(strip=True)
        match = re.search(r'(\d+)', text)
        item_id = match.group(1) if match else None
    else:
        item_id = None

    # Описание
    desc_div = soup.find('div', {'data-marker': 'item-view/item-description'})
    if desc_div:
        description = ' '.join(desc_div.stripped_strings)
    else:
        description = 'Описание не найдено'

    classified = {'itemId': item_id, 'description': description}
    print(classified)

except Exception as e:
    print(f'Ошибка: {e}')

In [ ]:
def clean_description(description):
    # Оставляем только:
    # - буквы: a-z A-Z а-я А-Я ё Ё
    # - цифры: 0-9
    # - пробелы и стандартные знаки препинания
    pattern = r'[^a-zA-Zа-яА-ЯёЁ0-9\s\.,!?;:()\[\]{}\'"-+=/&*#@$%~`_—-]'
    cleaned = re.sub(pattern, '', description)
    # Убираем лишние пробелы (множественные заменяем на один)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

# Применяем
description_raw = classified['description']
cleaned_description = clean_description(description_raw)

print(cleaned_description)

In [ ]:
def normalize_text(text):
    # 1. Приводим к нижнему регистру
    text = text.lower()
    
    # 2. Уменьшаем повторяющиеся буквы (3+ одинаковых подряд → 2)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Удаляем повторяющиеся знаки препинания
    text = re.sub(r'([.,!?;:"-+=/&*#@$%~`_—-]){2,}', r'\1', text)
    
    return text

# Применяем к cleaned_description
normalized_description = normalize_text(cleaned_description)

print(normalized_description)

In [ ]:
def split_into_chunks(text):
    # делим по всем основным разделителям
    chunks = re.split(r'[.!?;,:\(\)\[\]\{\}/\—\_]|\sи\s|\sили\s', text)

    # чистим пробелы и убираем пустые чанки
    clean_chunks = []

    for c in chunks:
        c = c.strip()
        if not c:
            continue

        # удалить только цифры
        if re.fullmatch(r'\d+', c):
            continue

        # удалить одиночные бесполезные слова
        if c in {"я", "мы"}:
            continue

        clean_chunks.append(c)

    return clean_chunks

# Применяем к нормализованному тексту
chunks_of_description = split_into_chunks(normalized_description)

print(chunks_of_description)

In [ ]:
# Загрузка словаря
def load_categories(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
    
# Подготовка embeddings для категорий
model = SentenceTransformer("deepvk/USER2-base")

def prepare_category_embeddings(categories):
    for cat in categories:
        cat["embeddings"] = model.encode(cat["keyPhrases"])
    return categories

# Функция similarity
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Матчинк одного чанка
def match_chunk(chunk, categories, threshold=0.75):
    chunk_emb = model.encode([chunk])[0]

    best_score = 0
    best_cat = None

    for cat in categories:
        for emb in cat["embeddings"]:
            score = cosine_sim(chunk_emb, emb)

            if score > best_score:
                best_score = score
                best_cat = cat

    if best_score >= threshold:
        return {
            "chunk": chunk,
            "mcId": best_cat["mcId"],
            "mcTitle": best_cat["mcTitle"],
            "score": float(best_score),
            "matched": True
        }

    return {
    "chunk": chunk,
    "mcId": best_cat["mcId"] if best_cat else None,
    "mcTitle": best_cat["mcTitle"] if best_cat else None,
    "score": float(best_score),
    "matched": False
    }

# Обработка всех чанков
def classify_chunks(chunks, categories):
    results = []

    for chunk in chunks:
        match = match_chunk(chunk, categories)
        results.append(match)

    return results

In [ ]:
categories = load_categories("temporary_test_dictionary.json")
categories = prepare_category_embeddings(categories)

chunk_classification = classify_chunks(chunks_of_description, categories)
matched = [c for c in chunk_classification if c["matched"]]
unmatched = [c for c in chunk_classification if not c["matched"]]

print("\n=== MATCHED ===")
for c in matched:
    print(c)

print("\n=== UNMATCHED ===")
for c in unmatched:
    print(c)

category_to_chunks = {}

for r in matched:
    mcId = r["mcId"]

    if mcId not in category_to_chunks:
        category_to_chunks[mcId] = []

    category_to_chunks[mcId].append(r["chunk"])

print('\n-------------------------------------------------------------\n')
print(category_to_chunks)

In [ ]:
# ---------- Загрузка NLI-модели ----------
model_checkpoint = 'cointegrated/rubert-base-cased-nli-threeway'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint)
if torch.cuda.is_available():
    model.cuda()
model.eval()

def predict_entailment(text: str, hypothesis: str) -> float:
    """Возвращает вероятность entailment (логического следования) от 0 до 1."""
    inputs = tokenizer(text, hypothesis, return_tensors='pt', truncation=True, max_length=512)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
    # У этой модели порядок классов: 0 - entailment, 1 - contradiction, 2 - neutral
    return probs[0]

# ---------- Основная функция для подготовки черновиков ----------
def prepare_drafts(cleaned_description: str,
                   category_to_chunks: Dict[int, List[str]],
                   categories: List[Dict],
                   threshold: float = 0.7,
                   debug: bool = True) -> Dict[str, Any]:
    # Если нет ни одного сопоставленного чанка — нечего обрабатывать
    if not category_to_chunks:
        if debug:
            print("Нет ни одного сопоставленного чанка. Черновики не создаются.")
        return {"shouldSplit": False, "drafts": []}
    
    title_by_id = {cat["mcId"]: cat["mcTitle"] for cat in categories}
    drafts = []
    if debug:
        print("\n===== DEBUG: анализ чанков =====")
    
    for mc_id, chunks in category_to_chunks.items():
        mc_title = title_by_id.get(mc_id, "Неизвестная категория")
        for chunk in chunks:
            hypothesis = f'В этом объявлении услуга "{chunk}" входит в комплекс работ.'
            prob = predict_entailment(cleaned_description, hypothesis)
            if debug:
                print(f"Чанк: '{chunk}' | Категория: {mc_title} (id={mc_id}) | Entailment prob: {prob:.4f} | Решение: {'отдельная услуга' if prob < threshold else 'часть комплекса'}")
            if prob < threshold:
                drafts.append({
                    "mcId": mc_id,
                    "mcTitle": mc_title,
                    "text": chunk
                })
    
    # Если есть отдельные услуги
    if drafts:
        should_split = True
    else:
        # Нет отдельных услуг, но чанки есть — значит все они входят в комплекс
        # Создаём один черновик для комплексной услуги
        complex_cat = None
        for cat in categories:
            if "ремонт под ключ" in cat["mcTitle"].lower():
                complex_cat = cat
                break
        if not complex_cat:
            # Если нет подходящей категории, создаём фиктивную
            complex_cat = {"mcId": 0, "mcTitle": "Комплексный ремонт"}
        
        # Собираем все чанки в один текст
        all_chunks = []
        for chunks in category_to_chunks.values():
            all_chunks.extend(chunks)
        combined_text = " ".join(all_chunks) if all_chunks else cleaned_description[:200]
        drafts.append({
            "mcId": complex_cat["mcId"],
            "mcTitle": complex_cat["mcTitle"],
            "text": combined_text
        })
        should_split = False
        if debug:
            print("\n===== Все услуги входят в комплекс. Создан черновик для комплексной услуги. =====")
    
    if debug:
        print(f"\nИтог: shouldSplit={should_split}, количество черновиков={len(drafts)}")
    return {"shouldSplit": should_split, "drafts": drafts}

In [ ]:
categories = load_categories("temporary_test_dictionary.json")
beta_response = prepare_drafts(cleaned_description, category_to_chunks, categories, threshold=0.7)
print(beta_response)

In [122]:
import csv
import json

def csv_to_json(csv_path, json_path, phrases_sep=','):
    data = []
    with open(csv_path, 'r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Разделяем keyPhrases по указанному разделителю, обрезаем пробелы, убираем пустые
            phrases = [p.strip() for p in row['keyPhrases'].split(phrases_sep) if p.strip()]
            data.append({
                "mcId": int(row['mcId']),
                "mcTitle": row['mcTitle'],
                "keyPhrases": phrases
            })
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Сконвертировано {len(data)} категорий в {json_path}")

# Использование
csv_to_json('rnc_mic_key_phrases.csv', 'rnc_mic_key_phrases.json', phrases_sep=';')

Сконвертировано 11 категорий в rnc_mic_key_phrases.json
